# TealKit MCP Agent - Weather Sensors MCP - Ministral Native Ollama Bootstrap Notebook

This notebook is the runnable bootstrap path for the `ollama_native` contract on the Ministral family.

It keeps the working Unsloth training and GGUF export flow from the current Ministral Colab notebook, but switches config, naming, and prompt loading to the native contract family.

Important: the repository's native contract is still a bootstrap contract. This notebook is runnable, but the final native runtime evaluation rules may still evolve.

## Cell 1 - Install Dependencies
Install Unsloth and the Colab-side training dependencies.

In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" trl peft accelerate bitsandbytes datasets huggingface_hub

import shutil
shutil.rmtree('/root/.unsloth', ignore_errors=True)

print('Install done. Restarting runtime...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## Cell 2 - Native Contract Config
Load the notebook variables for the `ollama_native` weather/Ministral path.

In [ ]:
from pathlib import Path

SERVER_SCOPE = 'weathersensorsmcp'
CONTRACT_TYPE = 'ollama_native'
MODEL_PRESET = 'ministral_3b'
MODEL_NAME = 'unsloth/Ministral-3-3B-Instruct-2512'
MODEL_SLUG = 'ministral-3b-weathersensorsmcp-ollama'
HF_REPO = f'lschaffer/{MODEL_SLUG}'  # change if needed
PROMPT_CONTRACT = 'contracts/ollama_native/prompt_contract.md'
QUALITY_GATE_PROFILE = 'weathersensorsmcp_ollama_native'

# ── Drive paths ───────────────────────────────────────────────────────────────
drive_root = Path('/content/drive/MyDrive/Tealkit/training') / SERVER_SCOPE
DATA_DIR = drive_root / 'datasets' / CONTRACT_TYPE / 'dataset_ministral'
TRAIN_FILE = DATA_DIR / 'train_split.jsonl'
VALID_FILE = DATA_DIR / 'valid_split.jsonl'

OUTPUT_DIR = drive_root / 'mcp_adapters_ministral_3b_ollama'
MERGE_DIR  = drive_root / 'mcp_merged_model_ministral_3b_ollama'
GGUF_DIR   = drive_root / 'mcp_fused_model_ministral_3b_ollama'

DRIVE_SYSTEM_PROMPT_FILE = DATA_DIR / 'ollama_native_system_prompt.md'

MAX_SEQ_LENGTH = 4096

DEFAULT_NATIVE_SYSTEM_PROMPT = (
    'You are the Cumulus Assistant for Weather Sensors MCP. '
    'Use tools accurately, keep tool-call turns concise, and keep final '
    'answers grounded in returned tool results.'
)
# System prompt is resolved in Cell 3 after Drive is mounted.
SYSTEM_PROMPT = DEFAULT_NATIVE_SYSTEM_PROMPT

print('SERVER_SCOPE :', SERVER_SCOPE)
print('CONTRACT_TYPE:', CONTRACT_TYPE)
print('MODEL_NAME   :', MODEL_NAME)
print('Train file   :', TRAIN_FILE)
print('Valid file   :', VALID_FILE)
print('HF repo      :', HF_REPO)


## Cell 3 - Mount Drive
Mount Google Drive and verify that the generated native dataset has been uploaded.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Resolve system prompt: Drive override > default
if DRIVE_SYSTEM_PROMPT_FILE.is_file():
    SYSTEM_PROMPT = DRIVE_SYSTEM_PROMPT_FILE.read_text(encoding='utf-8').strip()
    print('System prompt loaded from Drive:', DRIVE_SYSTEM_PROMPT_FILE)
else:
    SYSTEM_PROMPT = DEFAULT_NATIVE_SYSTEM_PROMPT
    print('System prompt: using default (no override on Drive)')

for check_path, label, required in [
    (TRAIN_FILE,               'train_split.jsonl',                   True),
    (VALID_FILE,               'valid_split.jsonl',                   True),
    (DRIVE_SYSTEM_PROMPT_FILE, 'drive native system prompt override', False),
]:
    if Path(check_path).is_file():
        print('OK      ', label, check_path)
    elif required:
        print('MISSING ', label, check_path)
    else:
        print('OPTIONAL', label, check_path)


## Cell 4 - Inspect Native Contract Inputs
Confirm the notebook is pointed at the native contract and not the text-only weather path.

In [ ]:
# Sanity-check: print the resolved paths and a sample from the train file.
import json

print('Train file :', TRAIN_FILE)
print('Valid file :', VALID_FILE)
print('Output dir :', OUTPUT_DIR)
print('GGUF dir   :', GGUF_DIR)
print()
print('System prompt:')
print(SYSTEM_PROMPT[:500])
print()

if TRAIN_FILE.is_file():
    with open(TRAIN_FILE, encoding='utf-8') as f:
        first = json.loads(f.readline())
    print('First train example keys:', list(first.keys()))
else:
    print('Train file not yet present — mount Drive and upload the splits first.')


## Cell 5 - Load Base Model
Use the same base Ministral family load path as the existing text-contract notebook.

In [ ]:
from unsloth import FastLanguageModel
import torch
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template='mistral',
)

print('Base model loaded.')

## Cell 6 - Apply LoRA
Keep the LoRA mechanics aligned with the text notebook and change only contract-specific data/validation behavior.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
model.print_trainable_parameters()

## Cell 7 - Load Dataset
The dataset should already be generated for the native contract before the notebook is used.

In [ ]:
import json
from datasets import Dataset, DatasetDict

def load_clean_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            # Normalize mixed types for tool calls to prevent PyArrow parse errors
            for msg in row.get('messages', []):
                if 'tool_calls' in msg:
                    for tc in msg['tool_calls']:
                        func = tc.get('function', {})
                        args = func.get('arguments', {})
                        if args:
                            # Normalize string-valued fields to str
                            for k in ['dlu_id', 'dluId']:
                                if k in args and args[k] is not None:
                                    args[k] = str(args[k])
                            for k in ['lat', 'lng', 'lon', 'km']:
                                if k in args and args[k] is not None:
                                    try:
                                        args[k] = float(args[k])
                                    except (ValueError, TypeError):
                                        pass
            data.append(row)
    return Dataset.from_list(data)

dataset = DatasetDict({
    'train': load_clean_jsonl(TRAIN_FILE),
    'validation': load_clean_jsonl(VALID_FILE)
})


def ensure_system_message(messages):
    if messages and isinstance(messages[0], dict) and messages[0].get('role') == 'system':
        return messages
    return [{'role': 'system', 'content': SYSTEM_PROMPT}] + list(messages)

def format_example(examples):
    texts = []
    for messages in examples['messages']:
        normalized = ensure_system_message(messages)
        texts.append(
            tokenizer.apply_chat_template(
                normalized, tokenize=False, add_generation_prompt=False
            )
        )
    return {'text': texts}

dataset = dataset.map(format_example, batched=True)

def detect_chat_markers(sample_text):
    instruction_candidates = [
        '[INST] ',
        '<|im_start|>user\n',
        '<|im_start|>user<|im_sep|>',
        '<|user|>',
    ]
    response_candidates = [
        ' [/INST]',
        '<|im_start|>assistant\n',
        '<|im_start|>assistant<|im_sep|>',
        '<|assistant|>',
    ]
    instruction_part = next((part for part in instruction_candidates if part in sample_text), None)
    response_part = next((part for part in response_candidates if part in sample_text), None)
    return instruction_part, response_part

sample_text = dataset['train'][0]['text'] if len(dataset['train']) else ''
INSTRUCTION_PART, RESPONSE_PART = detect_chat_markers(sample_text)

print('Train examples:', len(dataset['train']))
print('Valid examples:', len(dataset['validation']))
print('Instruction marker:', repr(INSTRUCTION_PART))
print('Response marker   :', repr(RESPONSE_PART))
print(sample_text[:2000])

## Cell 8 - Training Placeholder
This scaffold leaves the final native response representation explicit instead of pretending the native-format training target is already finished.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

trainer_args = SFTConfig(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=5,
    max_steps=250,
    learning_rate=5e-5,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='no',
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=3407,
    output_dir='/content/outputs',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=trainer_args,
)

if INSTRUCTION_PART and RESPONSE_PART:
    try:
        masked_trainer = train_on_responses_only(
            trainer,
            instruction_part=INSTRUCTION_PART,
            response_part=RESPONSE_PART,
        )
        if len(masked_trainer.train_dataset) == 0:
            print('WARNING: Response masking removed every train sample. Falling back to full-sequence training.')
        else:
            trainer = masked_trainer
            print(f'Response masking active: {len(trainer.train_dataset)} train samples.')
    except Exception as exc:
        print('WARNING: Response masking failed, using full sequence. Error:', exc)
else:
    print('WARNING: Could not detect chat markers in formatted text. Using full-sequence training.')

trainer.train()
print('Training complete.')

## Cell 9 - Export Placeholder
Keep the same broad save, merge, GGUF export, and Hugging Face upload stages as the existing Ministral notebook, but preserve the native model naming.

In [ ]:
import gc
import glob
import os
import shutil
import subprocess
import sys

# ── Ensure all Drive paths are plain strings (not PosixPath) ────────────────
OUTPUT_DIR   = str(OUTPUT_DIR)
MERGE_DIR    = str(MERGE_DIR)
GGUF_DIR     = str(GGUF_DIR)

QUANT_METHOD = 'q5_k_m'
GGUF_BASENAME = f"{HF_REPO.split('/')[-1]}-unsloth"
GGUF_F16_PATH = os.path.join(GGUF_DIR, f'{GGUF_BASENAME}-F16.gguf')
GGUF_QUANT_PATH = os.path.join(GGUF_DIR, f'{GGUF_BASENAME}-{QUANT_METHOD.upper()}.gguf')
FINAL_GGUF_FILE = None
GGUF_FILENAME = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Adapters successfully saved to Drive:', OUTPUT_DIR)

def run_checked(command, cwd=None, extra_env=None):
    env = os.environ.copy()
    for key in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'PYTHONUSERBASE'):
        env.pop(key, None)
    env['PYTHONNOUSERSITE'] = '1'
    if extra_env:
        for key, value in extra_env.items():
            if value is None:
                env.pop(key, None)
            else:
                env[key] = value
    # Coerce every element to str so PosixPath values don't crash join()
    command = [str(c) for c in command]
    print('>>', ' '.join(command))
    try:
        subprocess.run(command, cwd=cwd, env=env, check=True)
    except subprocess.CalledProcessError:
        # Re-run with stderr captured to show the actual error
        result = subprocess.run(command, cwd=cwd, env=env, capture_output=True, text=True)
        err = result.stderr
        if len(err) > 3000:
            print('=== STDERR (last 3000 chars) ===')
            print(err[-3000:])
        else:
            print('=== STDERR ===')
            print(err)
        raise

def clear_unsloth_llama_cpp_cache():
    cache_dir = '/root/.unsloth/llama.cpp'
    if os.path.isdir(cache_dir):
        shutil.rmtree(cache_dir, ignore_errors=True)
        print('Cleared cached Unsloth llama.cpp checkout:', cache_dir)

def refresh_tokenizer_files(merged_dir, base_model):
    from huggingface_hub import hf_hub_download
    for filename in ('tokenizer_config.json', 'tokenizer.json', 'special_tokens_map.json'):
        try:
            source_path = hf_hub_download(repo_id=base_model, filename=filename)
            shutil.copy2(source_path, os.path.join(merged_dir, filename))
        except Exception as exc:
            print(f'INFO: Could not refresh {filename}: {exc}')
    try:
        source_path = hf_hub_download(repo_id=base_model, filename='tokenizer.model')
        shutil.copy2(source_path, os.path.join(merged_dir, 'tokenizer.model'))
    except Exception:
        pass

def find_quantize_binary(llama_cpp_dir):
    candidates = [
        os.path.join(llama_cpp_dir, 'build', 'bin', 'llama-quantize'),
        os.path.join(llama_cpp_dir, 'build', 'bin', 'quantize'),
        shutil.which('llama-quantize'),
        shutil.which('quantize'),
    ]
    for candidate in candidates:
        if candidate and os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            return candidate
    return None

def ensure_quantize_binary(llama_cpp_dir):
    quantize_bin = find_quantize_binary(llama_cpp_dir)
    if quantize_bin:
        return quantize_bin
    run_checked(['cmake', '-S', '.', '-B', 'build', '-DBUILD_SHARED_LIBS=OFF', '-DGGML_CUDA=OFF'], cwd=llama_cpp_dir)
    run_checked(['cmake', '--build', 'build', '--config', 'Release', '-j2'], cwd=llama_cpp_dir)
    return find_quantize_binary(llama_cpp_dir)

def install_llama_cpp_requirements(llama_cpp_dir, pydeps_dir):
    requirements_file = os.path.join(llama_cpp_dir, 'requirements.txt')
    if os.path.isdir(pydeps_dir):
        shutil.rmtree(pydeps_dir)
    os.makedirs(pydeps_dir, exist_ok=True)
    if os.path.isfile(requirements_file):
        run_checked([sys.executable, '-m', 'pip', 'install', '--upgrade', '--target', pydeps_dir, '-r', requirements_file])

def ensure_llama_cpp_checkout(llama_cpp_dir):
    convert_script = os.path.join(llama_cpp_dir, 'convert_hf_to_gguf.py')
    if not os.path.isdir(os.path.join(llama_cpp_dir, '.git')):
        if os.path.exists(llama_cpp_dir):
            shutil.rmtree(llama_cpp_dir, ignore_errors=True)
        run_checked(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', llama_cpp_dir])
    return convert_script

def manual_llama_cpp_convert(merged_dir):
    clear_unsloth_llama_cpp_cache()
    refresh_tokenizer_files(merged_dir, MODEL_NAME)
    llama_cpp_dir = '/content/llama.cpp'
    pydeps_dir = '/content/llama_cpp_pydeps'
    convert_script = ensure_llama_cpp_checkout(llama_cpp_dir)
    gguf_py_dir = os.path.join(llama_cpp_dir, 'gguf-py')
    # Install llama.cpp gguf-py package into the Colab environment (not an isolated pydeps)
    # so the conversion script uses the same well-tested transformers version as Colab.
    run_checked([sys.executable, '-m', 'pip', 'install', '--upgrade', gguf_py_dir])
    quantize_bin = ensure_quantize_binary(llama_cpp_dir)
    # Run conversion without -S/isolated mode so it uses Colab's system transformers.
    run_checked([sys.executable, convert_script, merged_dir, '--outfile', GGUF_F16_PATH, '--outtype', 'f16'], cwd=llama_cpp_dir)
    if quantize_bin:
        run_checked([quantize_bin, GGUF_F16_PATH, GGUF_QUANT_PATH, QUANT_METHOD.upper()])
        return GGUF_QUANT_PATH
    return GGUF_F16_PATH

if os.path.exists(GGUF_DIR):
    shutil.rmtree(GGUF_DIR)
os.makedirs(GGUF_DIR, exist_ok=True)

try:
    clear_unsloth_llama_cpp_cache()
    trainer.model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=QUANT_METHOD)
    gguf_files = sorted(glob.glob(f'{GGUF_DIR}/*.gguf'))
    if not gguf_files:
        raise RuntimeError(f'No GGUF files were created in {GGUF_DIR}')
    FINAL_GGUF_FILE = gguf_files[0]
    GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)
    print('Native GGUF export complete:', FINAL_GGUF_FILE)
except Exception as exc:
    print('Native GGUF export failed, falling back to manual llama.cpp conversion:', exc)
    if os.path.exists(MERGE_DIR):
        shutil.rmtree(MERGE_DIR)
    try:
        trainer.model.save_pretrained_merged(MERGE_DIR, tokenizer, save_method='merged_16bit')
    except RuntimeError as merge_err:
        # Unsloth's post-merge LoRA count check can fire spuriously (# of LoRAs != # of saved modules)
        # after a successful merge — the merge itself produces .safetensors files.
        # Check if merge actually wrote files; if yes, proceed with GGUF conversion.
        merged_files = glob.glob(f'{MERGE_DIR}/*.safetensors') + glob.glob(f'{MERGE_DIR}/*.bin')
        if not merged_files:
            raise RuntimeError(f'Merge failed — no model weight files found in {MERGE_DIR}') from merge_err
        print(f'Merge count-check error ignored — {len(merged_files)} weight file(s) found in {MERGE_DIR}')
    FINAL_GGUF_FILE = manual_llama_cpp_convert(MERGE_DIR)
    GGUF_FILENAME = os.path.basename(FINAL_GGUF_FILE)

if not FINAL_GGUF_FILE or not os.path.isfile(FINAL_GGUF_FILE):
    raise RuntimeError('GGUF export failed: no .gguf file was created.')

print('Final GGUF file:', FINAL_GGUF_FILE)
torch.cuda.empty_cache()
gc.collect()

## Cell 10 - Native Evaluation
Validate this contract with the native Ollama quality-gate profile, which checks assistant tool_calls instead of the legacy text wrapper.

In [ ]:
MODEL_CARD_PATH = f'{GGUF_DIR}/README.md'
GGUF_FILENAME = globals().get('GGUF_FILENAME') or f"{HF_REPO.split('/')[-1]}-unsloth-{QUANT_METHOD.upper()}.gguf"

model_card_content = f'''---
base_model: {MODEL_NAME}
library_name: unsloth
tags:
- mcp
- weather-sensors
- tool-calling
- ollama-native
- gguf
---

# Weather Sensors MCP Agent - {MODEL_NAME.split('/')[-1]}

This model was fine-tuned for the native Ollama contract of Weather Sensors MCP.

## Files
- GGUF: {GGUF_FILENAME}
- Adapters: saved during notebook execution

## Contract
- Prompt contract: {PROMPT_CONTRACT}
- Quality gate profile: {QUALITY_GATE_PROFILE}
'''

with open(MODEL_CARD_PATH, 'w', encoding='utf-8') as handle:
    handle.write(model_card_content)
print('Model card generated at:', MODEL_CARD_PATH)

# ── Generate Modelfile (required by download-hf-model.sh for Ollama tool support) ──
MODELFILE_PATH = f'{GGUF_DIR}/Modelfile'
modelfile_content = f'''FROM {GGUF_FILENAME}

TEMPLATE """{{{{ if .System }}}}[INST] {{{{ .System }}}}

{{{{ .Prompt }}}} [/INST]{{{{ else }}}}[INST] {{{{ .Prompt }}}} [/INST]{{{{ end }}}}"""

PARAMETER stop "</s>"
PARAMETER num_ctx {MAX_SEQ_LENGTH}
PARAMETER num_thread 4
'''

with open(MODELFILE_PATH, 'w', encoding='utf-8') as handle:
    handle.write(modelfile_content)
print('Modelfile generated at:', MODELFILE_PATH)
print('Quality gate profile:', QUALITY_GATE_PROFILE)

UPLOAD_TO_HF = False  # set to True when you want to upload immediately

if UPLOAD_TO_HF:
    import getpass
    from huggingface_hub import HfApi, upload_folder
    try:
        hf_token = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
        api = HfApi(token=hf_token)
        api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
        upload_folder(
            repo_id=HF_REPO,
            folder_path=GGUF_DIR,
            repo_type='model',
            token=hf_token,
        )
        print('Uploaded GGUF folder to Hugging Face:', HF_REPO)
    except Exception as exc:
        print('HF upload skipped/failed:', exc)